In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import torch
import mlflow
import time

import logging
import warnings
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="neuralforecast")

import sys
import os
sys.path.insert(0, os.path.abspath("../../"))

from QNeuralForecast import QDLinear, QNHITS, QDeepAR, QTimesNet
from neuralforecast.models import DLinear, NHITS, DeepAR, TimesNet

from neuralforecast import NeuralForecast
from pytorch_lightning.loggers import MLFlowLogger
from neuralforecast.losses.numpy import mae, mse, rmse, mape, smape, mase

In [ ]:
mlflow.set_experiment("/Users/silas.schroeder@sap.com/qtsf")

In [ ]:
def set_global_seed(seed: int):
    """Fix all global random sources for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(nf_fitted):
    """Return {model_class_name: trainable_param_count} for every model in a fitted NF."""
    counts = {}
    for m in nf_fitted.models:
        name = m.__class__.__name__
        n = sum(p.numel() for p in m.parameters() if p.requires_grad)
        counts[name] = n
    return counts

In [ ]:
df = pd.read_csv("../data/train_prepared.csv", parse_dates=["Date"], low_memory=False)
df.drop(columns="id",inplace=True)
df = df.sort_values(["Store", "Date"])

delta_days = (max(df["Date"]) - min(df["Date"])).days
train_days = int(delta_days * 0.8)
split_date = min(df["Date"]) + np.timedelta64(train_days,"D")

train_df = df[df["Date"] < split_date].copy()
test_df  = df[df["Date"] >= split_date].copy()

HORIZON = (max(df["Date"]) - np.datetime64(split_date)).days

In [ ]:
static_cols = [
    'StoreType', 'Assortment', 'CompetitionDistance',
    'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear',
    'CompetitionOpenSince', 'Promo2Since'
]
futr_cols = [
    'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'DayOfWeek',
    'day_of_week', 'day_of_month', 'day_of_year', 'month', 'week_of_month',
    'week_of_year', 'year', 'quarter', 'is_month_start', 'is_month_end',
    'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos', 'doy_sin', 'doy_cos',
    'wom_sin', 'wom_cos', 'woy_sin', 'woy_cos', 'moy_sin', 'moy_cos',
    'promo_month_flag',
]
hist_cols = [
    'Sales', 'Customers',
    'lag_7', 'lag_14', 'lag_28', 'lag_customers_7',
    'rolling_mean_7', 'rolling_mean_28'
]
static_df = df.groupby('Store')[static_cols].first().reset_index()
futr_exog_df = df[['Store', 'Date'] + futr_cols].copy()

def exog_kwargs(cls):
    """Return only the exog lists the model class actually supports, based on its EXOGENOUS_* flags."""
    kwargs = {}
    if getattr(cls, "EXOGENOUS_HIST", False):
        kwargs["hist_exog_list"] = hist_cols
    if getattr(cls, "EXOGENOUS_FUTR", False):
        kwargs["futr_exog_list"] = futr_cols
    if getattr(cls, "EXOGENOUS_STAT", False):
        kwargs["stat_exog_list"] = static_cols
    return kwargs

In [ ]:
# ── Experiment label ─────────────────────────────────────────────────────────
# Change this before each run to keep results separated in W&B.
# e.g. "dev", "hpo_test", "final"
EXPERIMENT = "dev"

# --- Classical models (no circuit_device param) — trained once as baseline ---
CLASSICAL_MODEL_CLASSES = [DLinear, NHITS, DeepAR, TimesNet]

# --- Quantum models (have circuit_device param) — trained once per device ---
QUANTUM_MODEL_CLASSES = [QDLinear, QNHITS, QDeepAR, QTimesNet]

# --- Devices to compare ---
# Each key becomes a run label in results; value is passed as circuit_device=...
QUANTUM_DEVICES = {
    "ideal":          "default.qubit",   # noise-free statevector simulation
    "noisy_sim":      "default.mixed",   # mixed-state sim — add depolarising noise in circuit
}

# Combined for HPO (shared best_params applied to all model types)
model_classes = CLASSICAL_MODEL_CLASSES + QUANTUM_MODEL_CLASSES

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
# --- Hyperparameter search space ---
HYPERPARAM_GRID = {
    "input_size":    [100, 200, 400],
    "max_steps":     [50, 100, 300],
    "learning_rate": (1e-4, 1e-1),   # log-uniform range
    "scaler_type":   ["standard", "robust", "minmax", "identity"],
}

# Storage for best params per model and study objects
best_params_per_model = {}
all_studies = {}  # Store study objects for visualization

# ═══════════════════════════════════════════════════════════════════════════
# HPO: Individual optimization for each model class
# ═══════════════════════════════════════════════════════════════════════════
print("="*70)
print("HYPERPARAMETER OPTIMIZATION — Individual per model")
print("="*70)

for model_cls in model_classes:
    model_name = model_cls.__name__
    print(f"Optimizing: {model_name}")

    def objective(trial):
        set_global_seed(trial.number)
        input_size    = trial.suggest_categorical("input_size",    HYPERPARAM_GRID["input_size"])
        max_steps     = trial.suggest_categorical("max_steps",     HYPERPARAM_GRID["max_steps"])
        learning_rate = trial.suggest_float("learning_rate", *HYPERPARAM_GRID["learning_rate"], log=True)
        scaler_type   = trial.suggest_categorical("scaler_type",   HYPERPARAM_GRID["scaler_type"])

        # Create model with trial hyperparameters
        model_kwargs = {
            "h": HORIZON,
            "input_size": input_size,
            "max_steps": max_steps,
            "learning_rate": learning_rate,
            "scaler_type": scaler_type,
            "start_padding_enabled": True,
            "logger": False,
            "enable_progress_bar": False,
            "enable_model_summary": False,
            "accelerator": 'cpu',
            "random_seed": trial.number,
            **exog_kwargs(model_cls),
        }

        # Add circuit_device for quantum models
        if model_cls in QUANTUM_MODEL_CLASSES:
            model_kwargs["circuit_device"] = "default.qubit"  # Use ideal for HPO (faster)

        trial_model = model_cls(**model_kwargs)
        nf_trial = NeuralForecast(models=[trial_model], freq="D")

        # Cross-validation
        cv = nf_trial.cross_validation(
            train_df, n_windows=4, step_size=100,
            verbose=False, refit=False, static_df=static_df,
            id_col="Store", time_col="Date", target_col="Sales",
        )

        # Get model column (should be the only one besides Store, Date, Sales, cutoff)
        model_col = [c for c in cv.columns if c not in ["Store", "Date", "Sales", "cutoff"]][0]
        cv_mae = float(mae(cv["Sales"].values, cv[model_col].values))
        return cv_mae

    # Create study for this model
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name=f"rossmann_hpo_{model_name}",
    )

    # Optimize
    study.optimize(objective, n_trials=10, n_jobs=1, show_progress_bar=True)

    # Store best params and study object
    best_params_per_model[model_name] = study.best_params
    all_studies[model_name] = study

    print(f"\n✓ {model_name} — Best CV MAE: {study.best_value:.4f}")

print(f"\n{'='*70}")
print("HPO COMPLETE — Best parameters per model:")
print(f"{'='*70}")
for model_name, params in best_params_per_model.items():
    print(f"\n{model_name}:")
    for param, value in params.items():
        print(f"  {param:15s}: {value}")

In [ ]:
# --- Visualization: HPO results per model ---
n_models = len(all_studies)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 4*n_models))

# Handle indexing for single vs multiple models
if n_models == 1:
    axes = axes.reshape(1, -1)

for idx, (model_name, study) in enumerate(all_studies.items()):
    ax_left = axes[idx, 0]
    ax_right = axes[idx, 1]
    
    # Left plot: Optimization history (MAE over trials)
    trials = study.trials
    trial_numbers = [t.number for t in trials]
    trial_values = [t.value for t in trials]
    best_values = [min(trial_values[:i+1]) for i in range(len(trial_values))]
    
    ax_left.plot(trial_numbers, trial_values, 'o-', alpha=0.6, label='Trial MAE')
    ax_left.plot(trial_numbers, best_values, 'r-', linewidth=2, label='Best MAE')
    ax_left.set_xlabel('Trial')
    ax_left.set_ylabel('CV MAE')
    ax_left.set_title(f'{model_name} — Optimization History')
    ax_left.legend()
    ax_left.grid(True, alpha=0.3)
    
    # Right plot: Parameter importance (frequency of best values)
    param_data = {}
    for param in ['input_size', 'max_steps', 'scaler_type']:
        param_data[param] = [t.params.get(param) for t in trials if param in t.params]
    
    # Count occurrences in top 5 trials
    top_5_trials = sorted(trials, key=lambda t: t.value)[:5]
    param_counts = {}
    for param in ['input_size', 'max_steps', 'scaler_type']:
        counts = {}
        for t in top_5_trials:
            val = str(t.params.get(param, 'N/A'))
            counts[val] = counts.get(val, 0) + 1
        param_counts[param] = counts
    
    # Plot as horizontal bar chart
    y_pos = 0
    colors = plt.cm.Set3(range(len(param_counts)))
    for color_idx, (param, counts) in enumerate(param_counts.items()):
        for val, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            ax_right.barh(y_pos, count, color=colors[color_idx], alpha=0.7)
            ax_right.text(count + 0.1, y_pos, f'{param}={val}', va='center')
            y_pos += 1
        y_pos += 0.5  # Add spacing between parameter groups
    
    ax_right.set_xlabel('Frequency in Top 5 Trials')
    ax_right.set_title(f'{model_name} — Parameter Importance')
    ax_right.set_yticks([])
    ax_right.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('hpo_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved visualization to: hpo_results.png")

# --- Summary table ---
print(f"\n{'='*70}")
print("BEST HYPERPARAMETERS SUMMARY")
print(f"{'='*70}\n")

summary_df = pd.DataFrame(best_params_per_model).T
summary_df.index.name = "Model"

# Add best MAE to summary
summary_df['best_cv_mae'] = [all_studies[model].best_value for model in summary_df.index]

print(summary_df.to_string())

# Save to CSV for later reference
summary_df.to_csv("best_hyperparameters.csv")
print(f"\n✓ Saved to: best_hyperparameters.csv")

In [ ]:
# Load best hyperparameters from CSV (persisted from HPO cell)
HYPERPARAMS_FILE = "best_hyperparameters.csv"
if not os.path.exists("best_hyperparameters.csv"):
    raise FileNotFoundError(
        f"'{HYPERPARAMS_FILE}' not found. Run the HPO cell first to generate it."
    )
best_params_df = pd.read_csv(HYPERPARAMS_FILE, index_col="Model")
best_params_per_model = best_params_df.drop(columns=["best_cv_mae"], errors="ignore").to_dict(orient="index")

SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# all_training_times : {run_label -> [elapsed_sec_per_seed, ...]}
# all_param_counts   : {run_label -> {model_name -> n_params}}  (same for every seed)
all_training_times = {}
all_param_counts   = {}

# ── 1. Classical models ──────────────────────────────────────────────────────
if CLASSICAL_MODEL_CLASSES:
    print("═" * 60)
    print("CLASSICAL MODELS — Training")
    print("═" * 60)
    classical_times = []
    for seed in SEEDS:
        set_global_seed(seed)
        seed_models = []
        
        for cls in CLASSICAL_MODEL_CLASSES:
            model_name = cls.__name__
            # Get best params for this specific model
            params = best_params_per_model.get(model_name)
            
            model = cls(
                h=HORIZON,
                input_size=int(params["input_size"]),
                max_steps=int(params["max_steps"]),
                learning_rate=float(params["learning_rate"]),
                scaler_type=str(params["scaler_type"]),
                start_padding_enabled=True,
                random_seed=seed,
                early_stop_patience_steps=5,
                logger=None,
                enable_progress_bar=False,
                enable_model_summary=False,
                accelerator='cpu',
                **exog_kwargs(cls)
            )
            seed_models.append(model)
        
        nf = NeuralForecast(models=seed_models, freq="D")
        t0 = time.perf_counter()
        nf.fit(train_df, id_col="Store", time_col="Date", target_col="Sales", val_size=HORIZON, static_df=static_df)
        elapsed = time.perf_counter() - t0
        classical_times.append(elapsed)

        nf.save(path=f'./models/classical/{seed}', overwrite=True, save_dataset=True)

        if seed == SEEDS[0]:
            all_param_counts["classical"] = count_parameters(nf)

    all_training_times["classical"] = classical_times

# ── 2. Quantum models ────────────────────────────────────────────────────────
if QUANTUM_MODEL_CLASSES:
    for dev_label, dev_name in QUANTUM_DEVICES.items():
        print("\n" + "═" * 60)
        print(f"QUANTUM MODELS  |  device: {dev_label}  ({dev_name}) — Training")
        print("═" * 60)
        dev_times = []
        for seed in SEEDS:
            set_global_seed(seed)
            seed_models = []
            
            for cls in QUANTUM_MODEL_CLASSES:
                model_name = cls.__name__
                # Get best params for this specific model
                params = best_params_per_model.get(model_name)
                
                model = cls(
                    h=HORIZON,
                    input_size=int(params["input_size"]),
                    max_steps=int(params["max_steps"]),
                    learning_rate=float(params["learning_rate"]),
                    scaler_type=str(params["scaler_type"]),
                    start_padding_enabled=True,
                    random_seed=seed,
                    early_stop_patience_steps=5,
                    circuit_device=dev_name,
                    logger=None,
                    enable_progress_bar=False,
                    enable_model_summary=False,
                    accelerator='cpu',
                    **exog_kwargs(cls)
                )
                seed_models.append(model)
            
            nf = NeuralForecast(models=seed_models, freq="D")
            t0 = time.perf_counter()
            nf.fit(train_df, id_col="Store", time_col="Date", target_col="Sales", val_size=HORIZON, static_df=static_df)
            elapsed = time.perf_counter() - t0
            dev_times.append(elapsed)

            nf.save(path=f'./models/{dev_label}/{seed}', overwrite=True, save_dataset=True)

            if seed == SEEDS[0]:
                all_param_counts[dev_label] = count_parameters(nf)

        all_training_times[dev_label] = dev_times

print(f"\nTraining complete. Saved run labels: {list(all_training_times.keys())}")

In [ ]:
# all_seed_forecasts : {run_label -> [seed_fc_df, ...]}
all_seed_forecasts = {}

# Initialise timing/param dicts if this cell is run standalone (without training cell)
if "all_training_times" not in dir():
    all_training_times = {}
if "all_param_counts" not in dir():
    all_param_counts = {}

# ── 0. Naive baseline (lag-1: forecast[t] = actual[t-1]) ────────────────────
print("═" * 60)
print("NAIVE BASELINE  (lag-1: forecast[t] = actual[t-1])")
print("═" * 60)
all_dates = (
    pd.concat([train_df[["Store", "Date", "Sales"]],
               test_df [["Store", "Date", "Sales"]]])
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)
all_dates["Naive"] = all_dates.groupby("Store")["Sales"].shift(1)
naive_fc = (
    all_dates
    .merge(test_df[["Store", "Date"]], on=["Store", "Date"], how="inner")
    [["Store", "Date", "Naive"]]
    .reset_index(drop=True)
)
all_seed_forecasts["naive"] = [naive_fc]
all_training_times.setdefault("naive", [0.0])
all_param_counts  .setdefault("naive", {"Naive": 0})
print(f"  Forecast range: {naive_fc['Date'].min().date()} → {naive_fc['Date'].max().date()}")

# ── 1. Classical models — load & predict ────────────────────────────────────
if CLASSICAL_MODEL_CLASSES:
    print("\n" + "═" * 60)
    print("CLASSICAL MODELS — Forecasting")
    print("═" * 60)
    classical_forecasts = []
    for seed in SEEDS:
        nf = NeuralForecast.load(f'./models/classical/{seed}')
        fc = nf.predict(futr_df=futr_exog_df, static_df=static_df).reset_index()
        classical_forecasts.append(fc)
    all_seed_forecasts["classical"] = classical_forecasts

# ── 2. Quantum models — load & predict ──────────────────────────────────────
if QUANTUM_MODEL_CLASSES:
    for dev_label, dev_name in QUANTUM_DEVICES.items():
        print("\n" + "═" * 60)
        print(f"QUANTUM MODELS  |  device: {dev_label} — Forecasting")
        print("═" * 60)
        dev_forecasts = []
        for seed in SEEDS:
            for cls in QUANTUM_MODEL_CLASSES:
                model_name = cls.__name__
                nf = NeuralForecast.load(f'./models/{dev_label}/{seed}')
                fc = nf.predict(futr_df=futr_exog_df, static_df=static_df).reset_index()
                dev_forecasts.append(fc)
        all_seed_forecasts[dev_label] = dev_forecasts

In [ ]:
y_train_global = train_df["Sales"].values
SEASONALITY = 7

# all_avg_results  : {run_label -> {model -> {metric -> float}}}  (seed-ensemble score)
# all_seed_results : {run_label -> {model -> {metric -> [per-seed floats]}}}
all_avg_results  = {}
all_seed_results = {}
all_avg_forecasts = {}  # {run_label -> avg_forecast_df}  — kept for store plots

for run_label, seed_forecasts in all_seed_forecasts.items():

    # --- Average forecasts across seeds ---
    model_names = [c for c in seed_forecasts[0].columns if c not in ["index", "Store", "Date"]]
    avg_forecasts = seed_forecasts[0][["Store", "Date"]].copy()
    for model in model_names:
        avg_forecasts[model] = np.mean([fc[model].values for fc in seed_forecasts], axis=0)
    all_avg_forecasts[run_label] = avg_forecasts

    comparison = test_df[["Store", "Date", "Sales"]].merge(
        avg_forecasts, on=["Store", "Date"], how="inner"
    )

    # --- Per-seed metrics ---
    seed_results = {m: {k: [] for k in ["MAE", "MSE", "RMSE", "MAPE", "MASE"]}
                    for m in model_names}
    for fc in seed_forecasts:
        comp_s = test_df[["Store", "Date", "Sales"]].merge(fc, on=["Store", "Date"], how="inner")
        for model in model_names:
            seed_results[model]["MAE" ].append(mae( comp_s["Sales"], comp_s[model]))
            seed_results[model]["MSE" ].append(mse( comp_s["Sales"], comp_s[model]))
            seed_results[model]["RMSE"].append(rmse(comp_s["Sales"], comp_s[model]))
            seed_results[model]["MAPE"].append(mape(comp_s["Sales"], comp_s[model]))
            seed_results[model]["MASE"].append(mase(comp_s["Sales"], comp_s[model],
                                                    y_train=y_train_global,
                                                    seasonality=SEASONALITY))

    # --- Ensemble (seed-averaged forecast) metrics ---
    avg_results = {}
    for model in model_names:
        avg_results[model] = {
            "MAE" : mae( comparison["Sales"], comparison[model]),
            "MSE" : mse( comparison["Sales"], comparison[model]),
            "RMSE": rmse(comparison["Sales"], comparison[model]),
            "MAPE": mape(comparison["Sales"], comparison[model]),
            "MASE": mase(comparison["Sales"], comparison[model],
                         y_train=y_train_global, seasonality=SEASONALITY),
        }

    all_avg_results [run_label] = avg_results
    all_seed_results[run_label] = seed_results

    # --- Training-time stats for this run label ---
    run_times = all_training_times.get(run_label, [])
    mean_training_time = float(np.mean(run_times)) if run_times else 0.0
    std_training_time  = float(np.std(run_times))  if run_times else 0.0

    # --- MLflow logging (one nested run per model x device combination) ---
    for model in model_names:
        with mlflow.start_run(
            run_name=f"{EXPERIMENT}/{run_label}/{model}",
            tags={"experiment": EXPERIMENT, "run_label": run_label, "model": model},
        ):
            # Log identifying parameters
            mlflow.log_params({
                "model":      model,
                "run_label":  run_label,
                "experiment": EXPERIMENT,
            })

            # Accuracy metrics (seed-ensemble and per-seed statistics)
            metrics: dict = {}
            for metric_name, val in avg_results[model].items():
                metrics[f"avg_{metric_name}"] = float(val)
            for metric_name, vals in seed_results[model].items():
                metrics[f"mean_{metric_name}"] = float(np.mean(vals))
                metrics[f"std_{metric_name}"]  = float(np.std(vals))

            # Model size
            param_count = all_param_counts.get(run_label, {}).get(model, None)
            if param_count is not None:
                metrics["num_parameters"] = float(param_count)

            # Training time (shared cost across all models in this run's seed loop)
            metrics["mean_training_time_s"] = mean_training_time
            metrics["std_training_time_s"]  = std_training_time

            mlflow.log_metrics(metrics)

In [ ]:
# Save forecasts for evaluation.ipynb
FORECASTS_DIR = "./forecasts"

for run_label, forecast_df in all_avg_forecasts.items():
    save_path = f"{FORECASTS_DIR}/{run_label}"
    os.makedirs(save_path, exist_ok=True)
    forecast_df.to_pickle(f"{save_path}/forecasts.pkl")
    print(f"✓ Saved {run_label} forecasts to {save_path}/forecasts.pkl")

print(f"\nAll forecasts saved. Total run labels: {len(all_avg_forecasts)}")